# Tệp: notebooks/01_data_quality_eda.ipynb
# Mục đích: Đánh giá chất lượng dữ liệu và khám phá MOOC-Radar bằng tiếng Việt, độc lập với huấn luyện.

# Đánh giá chất lượng dữ liệu và EDA — MOOC-Radar

Notebook đọc dữ liệu gốc theo từng chuỗi sinh viên, không sửa hoặc xóa bản ghi. Kết quả được lưu trong một thư mục `artifacts/eda_<run_id>/` mới để dùng cho báo cáo Modern MIS. Chạy notebook này **trước** bước chuẩn hóa và huấn luyện.


## 1. Mục tiêu phân tích

Kiểm tra cấu trúc file, missing values, trùng lặp, timestamp, liên kết problem–concept–course và độ bao phủ nhãn Bloom/video. Các biểu đồ mô tả dữ liệu quan sát; chúng không chứng minh nguyên nhân hay hiệu quả học tập. Chế độ `sample` thuận tiện để thử nhanh, còn `full` dùng toàn bộ file fine cục bộ.


In [ ]:
from pathlib import Path
import csv
import json
import os
import subprocess
import sys

from IPython.display import Image, Markdown, display
import pandas as pd


## 2. Thư viện và cấu hình

Trên Colab, notebook mount Google Drive bằng cơ chế chính thức và cài dependencies từ dự án. Có thể đặt biến môi trường `MOOC_PROJECT_ROOT`, `MOOC_DATA_ROOT`, `MOOC_ARTIFACT_ROOT`; không cần sửa mã nguồn. Khi chạy local, notebook tự tìm thư mục gốc dự án từ thư mục hiện tại.


In [ ]:
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    drive.mount("/content/drive")

local_candidates = [Path.cwd(), *Path.cwd().parents]
local_root = next((p for p in local_candidates if (p / "src/moocradar").is_dir()), None)
default_root = Path("/content/drive/MyDrive/Personalized_learning_path") if IN_COLAB else local_root
if default_root is None and "MOOC_PROJECT_ROOT" not in os.environ:
    raise FileNotFoundError("Đặt MOOC_PROJECT_ROOT tới thư mục dự án")
PROJECT_ROOT = Path(os.environ.get("MOOC_PROJECT_ROOT", str(default_root))).expanduser()
DATA_ROOT = Path(os.environ.get("MOOC_DATA_ROOT", str(PROJECT_ROOT / "data/raw"))).expanduser()
ARTIFACT_ROOT = Path(os.environ.get("MOOC_ARTIFACT_ROOT", str(PROJECT_ROOT / "artifacts"))).expanduser()
MODE = "sample"  # Đổi thành "full" để phân tích toàn bộ file fine.
SAMPLE_STUDENTS = 25
assert (PROJECT_ROOT / "src/moocradar/eda.py").is_file(), "Kiểm tra PROJECT_ROOT"
assert (DATA_ROOT / "problem.json").is_file(), "Kiểm tra DATA_ROOT"


In [ ]:
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(PROJECT_ROOT / "requirements.txt")], check=True)
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from moocradar.eda import inspect_files, inspect_schema, run_eda
print("Chế độ:", MODE, "| Dữ liệu:", DATA_ROOT)


## 3. Khảo sát file và cấu trúc dữ liệu

Bảng đầu chỉ đọc metadata của file; bảng sau lấy **một** sự kiện mẫu từ từng JSON array và chỉ in tên trường/kiểu dữ liệu. Không hiển thị `user_id`, `log_id` hay nội dung câu hỏi dài. `problem.json` là JSON Lines; ba file tương tác là JSON array chứa các chuỗi `seq`.


In [ ]:
file_table = pd.DataFrame(inspect_files(DATA_ROOT))
display(file_table)
display(pd.DataFrame(inspect_schema(DATA_ROOT)))


## 4. Tổng quan dataset

Lệnh dưới đây chạy EDA trực tiếp trên raw data bằng loader dùng chung. `sample` đọc 25 chuỗi đầu tiên của file fine nhưng vẫn đọc đầy đủ problem metadata; `full` quét toàn bộ file fine. Mỗi lần chạy tạo `run_id` mới và không cần checkpoint mô hình.


In [ ]:
max_students = SAMPLE_STUDENTS if MODE == "sample" else None
run_dir = run_eda(DATA_ROOT, ARTIFACT_ROOT, max_students=max_students)
report = json.loads((run_dir / "data_quality_report.json").read_text(encoding="utf-8"))
print("Run EDA:", run_dir.name, "| Phạm vi:", report["mode"])


In [ ]:
overview_keys = ["problem_records", "student_sequences", "interactions", "users", "interacted_problems", "courses", "concepts"]
overview = [{"Chỉ số": key, "Giá trị": report["counts"].get(key, 0)} for key in overview_keys]
display(pd.DataFrame(overview))
display(pd.DataFrame([report["sequence_length"]]).rename(columns={"min": "Ngắn nhất", "median": "Trung vị", "max": "Dài nhất"}))
display(pd.DataFrame(report["schema"]))
display(Markdown(f"**Nhận xét:** Đã quét {report['counts']['interactions']:,} lượt trả lời của {report['counts']['users']:,} sinh viên trong chế độ `{MODE}`. Số bài có tương tác có thể nhỏ hơn số problem metadata vì không phải bài nào cũng được làm trong phạm vi quét."))


## 5. Data Quality Assessment

Missing values được tính trên số lượt/bài đã quét, không tự điền hoặc xóa. Bản ghi trùng được kiểm tra bằng `log_id` và toàn bộ nội dung sự kiện; một problem nối nhiều concept là quan hệ dữ liệu hợp lệ, không phải câu trả lời trùng.


In [ ]:
missing = pd.read_csv(run_dir / "missing_values.csv")
display(missing.sort_values(["dataset", "missing_pct"], ascending=[True, False]))
score_row = missing.query("dataset == 'interactions' and field == 'score'").iloc[0]
display(Markdown(f"**Nhận xét:** `score` thiếu {int(score_row.missing_count):,} lượt ({score_row.missing_pct:.2f}%). Đây là dữ liệu điểm gốc; nhãn `is_correct` được đánh giá riêng."))


In [ ]:
quality_keys = ["duplicate_log_id", "duplicate_exact_record", "out_of_order_sequences", "out_of_order_pairs", "invalid_submit_time", "invalid_is_correct", "invalid_attempts", "invalid_score", "score_above_problem_max", "unmatched_problem_events", "events_without_concepts", "multi_concept_problems"]
quality = [{"Kiểm tra": key, "Số lượng": report["counts"].get(key, 0)} for key in quality_keys]
display(pd.DataFrame(quality))
display(Markdown(f"**Nhận xét:** Có {report['counts'].get('out_of_order_sequences', 0):,} chuỗi sai thứ tự thời gian gốc. Bước preprocessing cần sắp xếp ổn định trước khi chia dữ liệu theo thời gian; notebook chỉ ghi nhận vấn đề."))


In [ ]:
coverage = pd.DataFrame([{"Chỉ số": key, "Tỷ lệ (%)": value} for key, value in report["coverage"].items()])
display(coverage)
print("Trường video quan sát được:", report["video_fields"])
display(Markdown("**Nhận xét:** Bloom được đánh giá từ nhãn problem nối sang lượt trả lời. Chỉ coi video có liên kết khi metadata có trường video tường minh; không suy ra video từ `context_id`."))


## 6. Exploratory Data Analysis

Các hình PNG được xuất với độ phân giải 220 dpi, nhãn tiếng Việt và nhận xét được tạo từ số liệu của chính run. Khi xem tần suất theo concept, một lượt trả lời có thể đóng góp cho nhiều concept. Biểu đồ tỷ lệ sai dùng ngưỡng số lượt tối thiểu để hạn chế nhiễu ở concept quá hiếm.


In [ ]:
plot_dir = run_dir / "plots/eda"
for name, note in list(report["plot_notes"].items())[:4]:
    display(Markdown(f"### {name}"))
    display(Image(filename=str(plot_dir / name), width=850))
    display(Markdown(note))


In [ ]:
for name, note in list(report["plot_notes"].items())[4:]:
    display(Markdown(f"### {name}"))
    display(Image(filename=str(plot_dir / name), width=850))
    display(Markdown(note))


## 7. Tổng hợp các phát hiện chính

Báo cáo sau được tạo từ dữ liệu đã quét. Nếu đang dùng chế độ mẫu, không khái quát tỷ lệ tương tác cho toàn bộ sinh viên; chuyển sang `full` trước khi trích dẫn số liệu tổng thể trong báo cáo môn học.


In [ ]:
summary_text = (run_dir / "eda_summary.md").read_text(encoding="utf-8")
display(Markdown(summary_text.split("## Khuyến nghị preprocessing")[0]))


## 8. Khuyến nghị làm sạch và preprocessing

Đây là đề xuất xử lý, chưa phải hành động xóa/sửa dữ liệu trong notebook. Các quyết định modeling cần giữ quan hệ nhiều concept và ngăn nhãn tương lai đi vào feature lịch sử.


In [ ]:
recommendations = summary_text.split("## Khuyến nghị preprocessing")[1].split("## Giới hạn diễn giải")[0]
display(Markdown("## Khuyến nghị preprocessing" + recommendations))


## 9. Xuất báo cáo và biểu đồ

Thư mục run chứa báo cáo JSON, bảng CSV, tóm tắt Markdown và các hình dưới `plots/eda/`. Có thể sao chép cả thư mục này cho thành viên làm dashboard hoặc đưa hình vào slide. File raw không bị chỉnh sửa.


In [ ]:
required = ["manifest.json", "data_quality_report.json", "missing_values.csv", "dataset_statistics.csv", "concept_statistics.csv", "course_statistics.csv", "monthly_statistics.csv", "eda_summary.md"]
assert all((run_dir / name).is_file() for name in required)
assert list(plot_dir.glob("*.png")), "Chưa có biểu đồ"
for path in sorted(run_dir.rglob("*")):
    if path.is_file():
        print(path.relative_to(run_dir))
